# Laboratorio Práctico: Introducción a API

Tiempo estimado necesario: **15** minutos

## Objetivos

Después de completar este laboratorio podrás:

*   Crear y usar APIs en Python


### Introducción

Una API permite que dos piezas de software se comuniquen entre sí. Al igual que una función, no tienes que saber cómo funciona la API, solo sus entradas y salidas. Un tipo esencial de API es una API REST que te permite acceder a recursos a través de Internet. En este laboratorio, revisaremos la biblioteca Pandas en el contexto de una API, también revisaremos una API REST básica.


## Table of Contents

<div class="alert alert-block alert-info" style="margin-top: 20px">
<li><a href="#Pandas-is-an-API">Pandas es una API</a></li>
<li><a href="#REST-APIs">APIs REST</a></li>
<li><a href="#Quiz">Cuestionario</a></li>

</div>

<hr>


## Pandas es una API


Pandas es en realidad un conjunto de componentes de software, gran parte de los cuales ni siquiera están escritos en Python.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

Creas un diccionario, esto es solo datos.


In [ ]:
dict_={'a':[11,21,31],'b':[12,22,32]}

Cuando creas un objeto Pandas con el constructor de dataframe, en la jerga de API esto es una "instancia". Los datos en el diccionario se pasan a la API de pandas. Luego usas el dataframe para comunicarte con la API.


In [ ]:
df=pd.DataFrame(dict_)
type(df)

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0101EN-SkillsNetwork/labs/Module%205/images/pandas_api.png" width="800," align="center" alt="logistic regression block diagram">


Cuando llamas al método `head`, el dataframe se comunica con la API mostrando las primeras filas del dataframe.


In [ ]:
df.head()

Cuando llamas al método `mean`, la API calculará la media y devolverá el valor.


In [ ]:
df.mean()

## APIs REST


<p>Las APIs REST funcionan enviando una <b>solicitud</b> (request), la solicitud se comunica a través de un mensaje HTTP. El mensaje HTTP generalmente contiene un archivo JSON. Esto contiene instrucciones sobre qué operación nos gustaría que el servicio o <b>recurso</b> realice. De manera similar, la API devuelve una <b>respuesta</b> (response), a través de un mensaje HTTP, esta respuesta generalmente se encuentra dentro de un JSON.</p>
<p>En este laboratorio, usaremos la <a href=https://pypi.org/project/nba-api/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkPY0101ENSkillsNetwork19487395-2021-01-01>API de la NBA</a> para determinar qué tan bien se desempeñaron los Golden State Warriors contra los Toronto Raptors. Usaremos la API para determinar la cantidad de puntos por los que los Golden State Warriors ganaron o perdieron en cada juego. Entonces, si el valor es tres, los Golden State Warriors ganaron por tres puntos. De manera similar, si los Golden State Warriors perdieron por dos puntos, el resultado será negativo dos. La API manejará muchos de los detalles, como los Endpoints y la Autenticación. </p>


Es bastante simple usar la api de la nba para hacer una solicitud de un equipo específico. No requerimos un JSON, todo lo que requerimos es una identificación (id). Esta información se almacena localmente en la API. Importamos el módulo `teams`.


In [ ]:
%pip install nba_api

In [ ]:
from nba_api.stats.static import teams
import matplotlib.pyplot as plt

La función one_dict() toma una lista de diccionarios (cada uno representando los detalles de un equipo) y los combina en un solo diccionario donde cada clave contiene una lista de todos los valores correspondientes.


In [ ]:
def one_dict(list_dict):
    keys=list_dict[0].keys()
    out_dict={key:[] for key in keys}
    for dict_ in list_dict:
        for key, value in dict_.items():
            out_dict[key].append(value)
    return out_dict

El método <code>get_teams()</code> devuelve una lista de diccionarios.


In [ ]:
nba_teams = teams.get_teams()

La clave de diccionario id tiene un identificador único para cada equipo como valor. Veamos los primeros tres elementos de la lista:


In [ ]:
nba_teams[0:3]

Para facilitar las cosas, podemos convertir el diccionario en una tabla. Primero, usamos la función <code>one dict</code>, para crear un diccionario. Usamos las claves comunes para cada equipo como las claves, el valor es una lista; cada elemento de la lista corresponde a los valores para cada equipo.
Luego convertimos el diccionario en un dataframe, cada fila contiene la información para un equipo diferente.


In [ ]:
dict_nba_team=one_dict(nba_teams)
df_teams=pd.DataFrame(dict_nba_team)
df_teams.head()

Usaremos el apodo del equipo para encontrar la identificación única, podemos ver la fila que contiene a los warriors usando la columna nickname de la siguiente manera:


In [ ]:
df_warriors=df_teams[df_teams['nickname']=='Warriors']
df_warriors

Podemos usar la siguiente línea de código para acceder a la primera columna del DataFrame:


In [ ]:
id_warriors=df_warriors[['id']].values[0][0]
# we now have an integer that can be used to request the Warriors information 
id_warriors

La función "League Game Finder " hará una llamada a la API, está en el módulo <code>stats.endpoints</code>.


In [ ]:
from nba_api.stats.endpoints import leaguegamefinder

El parámetro <code>team_id_nullable</code> es la identificación única para los warriors. Bajo el capó, la API de la NBA está haciendo una solicitud HTTP.\
La información solicitada se proporciona y se transmite a través de una respuesta HTTP que se asigna al objeto <code>game finder</code>.


In [ ]:
# https://stats.nba.com no permite llamadas a la API desde IPs en la nube.

gamefinder = leaguegamefinder.LeagueGameFinder(team_id_nullable=id_warriors)

Podemos ver el archivo json ejecutando la siguiente línea de código.


In [ ]:
gamefinder.get_json()

El objeto game finder tiene un método <code>get_data_frames()</code>, que devuelve un dataframe. Si vemos el dataframe, podemos ver que contiene información sobre todos los juegos que jugaron los Warriors. La columna <code>PLUS_MINUS</code> contiene información sobre el puntaje, si el valor es negativo, los Warriors perdieron por esa cantidad de puntos, si el valor es positivo, los warriors ganaron por esa cantidad de puntos. La columna <code>MATCHUP</code> tiene el equipo contra el que jugaban los Warriors, GSW significa Golden State Warriors y TOR significa Toronto Raptors. <code>vs</code> significa que fue un juego en casa y el símbolo <code>@ </code> significa un juego fuera de casa.


In [ ]:
games = gamefinder.get_data_frames()[0]
games.head()

Puedes descargar el dataframe de la llamada a la API para Golden State y ejecutar el resto como un video.


In [ ]:
import requests

filename = "https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/PY0101EN/Chapter%205/Labs/Golden_State.pkl"

def download(url, filename):
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)

download(filename, "Golden_State.pkl")


In [ ]:
file_name = "Golden_State.pkl"
games = pd.read_pickle(file_name)
games.head()

Podemos crear dos dataframes, uno para los juegos en los que los Warriors se enfrentaron a los raptors en casa, y el segundo para los juegos fuera de casa.


In [ ]:
games_home=games[games['MATCHUP']=='GSW vs. TOR']
games_away=games[games['MATCHUP']=='GSW @ TOR']

Podemos calcular la media para la columna <code>PLUS_MINUS</code> para los dataframes <code>games_home</code> y <code> games_away</code>:


In [ ]:
games_home['PLUS_MINUS'].mean()

In [ ]:
games_away['PLUS_MINUS'].mean()

Podemos trazar la columna <code>PLUS MINUS</code> para los dataframes <code>games_home</code> y <code> games_away</code>.
Vemos que los warriors jugaron mejor en casa.


In [ ]:
fig, ax = plt.subplots()

games_away.plot(x='GAME_DATE',y='PLUS_MINUS', ax=ax)
games_home.plot(x='GAME_DATE',y='PLUS_MINUS', ax=ax)
ax.legend(["away", "home"])
plt.show()

## Cuestionario


Calcula la media para la columna <code>PTS</code> para los dataframes <code>games_home</code> y <code> games_away</code>:


In [ ]:
# Escribe tu código abajo y presiona Shift+Enter para ejecutar


<details><summary>Haz clic aquí para ver la solución</summary>

```python
games_home['PTS'].mean()

games_away['PTS'].mean()

```

</details>
